# 07 - Bridging Loan LGD Model

**Exit-Driven Bridging LGD - Australian Bank-Style Portfolio Framework**

Bridging loan LGD is primarily an exit-risk problem. This notebook demonstrates a pragmatic, auditable framework around exit pathway uncertainty.

| Step | Description |
|---|---|
| 1 | Generate synthetic bridging loan default dataset |
| 2 | Build base LGD from exit drivers |
| 3 | Show delay effect on LGD |
| 4 | Apply failed-exit stress scenarios |
| 5 | Produce EAD-weighted outputs and exports |


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(49)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 90)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")



## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** transparent proxy assumptions are used where observed internal bridging workout tapes are unavailable.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status (standard policy):** this is a portfolio-project proxy baseline, **not** internally calibrated to real workout outcomes.
- **Output standard:** report `lgd_base`, `lgd_downturn`, `lgd_final`, EAD-weighted outputs, and base recovery-time metric (`exit_time_base`).


## Objective
Build an interview-ready bridging LGD module where severity is governed by exit risk and weighted outputs.

## Drivers
- Exit type (refinance vs sale)
- Exit certainty
- Valuation risk
- Time to exit and failed-exit probability

## Logic
- Base/downturn/final LGD with delay and failure overlays and weighted aggregation

## Downturn
- Exit delays and failed-refinance channels driving higher liquidation severity

## Validation
- Weighted scenario/segment/delay outputs and validation checks

## Limitations
- Synthetic portfolio and proxy assumptions; future calibration requires internal workout data


## Why Bridging Risk = Exit Risk

Bridging loans are short-tenor and rely on a credible exit. Default severity is driven by **how and when the exit occurs**:

- Exit type (`refinance` vs `sale`)
- Exit certainty (probability of successful execution)
- Valuation risk at exit
- Time to exit

Delay and failed exits usually convert refinance assumptions into forced-sale outcomes, raising LGD materially.


## 1. Generate Synthetic Bridging Loan Dataset


In [ ]:
n_cases = 230
rng = np.random.default_rng(49)

purposes = ['acquisition_bridge', 'construction_refi_bridge', 'residual_stock_exit_bridge']
purpose_weights = [0.40, 0.35, 0.25]

rows = []
for i in range(1, n_cases + 1):
    purpose = rng.choice(purposes, p=purpose_weights)

    gross_value = float(rng.uniform(2.5e6, 65e6))
    leverage = float(np.clip(rng.normal(0.72, 0.08), 0.45, 0.92))
    ead = gross_value * leverage * rng.uniform(0.98, 1.04)

    exit_type_initial = rng.choice(['refinance', 'sale'], p=[0.62, 0.38])
    exit_certainty = float(np.clip(rng.normal(0.63, 0.18), 0.05, 0.98))
    valuation_risk = float(np.clip(rng.normal(0.38, 0.20), 0.03, 0.95))
    time_to_exit_months = float(np.clip(rng.normal(8.0, 3.0), 2.0, 24.0))

    contract_rate_proxy = float(np.clip(rng.normal(0.082, 0.014), 0.050, 0.125))
    cost_of_funds_proxy = float(np.clip(rng.normal(0.060, 0.011), 0.035, 0.095))
    discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)

    failure_prob_base = float(np.clip(
        0.06
        + 0.50 * (1 - exit_certainty)
        + 0.26 * valuation_risk
        + 0.20 * ((time_to_exit_months - 6) / 18),
        0.01,
        0.95,
    ))
    failed_exit_flag = int(rng.uniform() < failure_prob_base)

    final_exit_type = 'sale' if (failed_exit_flag == 1 and exit_type_initial == 'refinance') else exit_type_initial
    time_to_exit_effective_months = float(np.clip(time_to_exit_months + failed_exit_flag * rng.uniform(4, 14), 2, 36))

    rows.append({
        'loan_id': f'BRG{i:04d}',
        'purpose': purpose,
        'ead': ead,
        'gross_value': gross_value,
        'exit_type_initial': exit_type_initial,
        'exit_certainty': exit_certainty,
        'valuation_risk': valuation_risk,
        'time_to_exit_months': time_to_exit_months,
        'time_to_exit_effective_months': time_to_exit_effective_months,
        'failure_prob_base': failure_prob_base,
        'failed_exit_flag': failed_exit_flag,
        'final_exit_type': final_exit_type,
        'contract_rate_proxy': contract_rate_proxy,
        'cost_of_funds_proxy': cost_of_funds_proxy,
        'discount_rate': discount_rate,
    })

bridge = pd.DataFrame(rows)

print('Bridging cases:', bridge.shape)
display(bridge['purpose'].value_counts())
display(bridge['final_exit_type'].value_counts())
bridge.head()



## 2. Base LGD from Exit Drivers


In [ ]:
def compute_bridging_lgd(df, delay_multiplier=1.0, valuation_addon=0.0, failure_addon=0.0, rate_addon=0.0):
    x = df.copy()

    exit_time = (x['time_to_exit_effective_months'] * delay_multiplier).clip(2, 42)
    valuation_risk = (x['valuation_risk'] + valuation_addon).clip(0.0, 1.0)
    fail_prob = (x['failure_prob_base'] + failure_addon).clip(0.0, 1.0)
    disc_rate = (x['discount_rate'] + rate_addon).clip(0.03, 0.20)

    is_sale = (x['final_exit_type'] == 'sale').astype(int)

    lgd_base = (
        0.10
        + 0.28 * (1 - x['exit_certainty'])
        + 0.22 * valuation_risk
        + 0.18 * ((exit_time - 4) / 20).clip(0, 1)
        + 0.10 * is_sale
        + 0.28 * fail_prob
    ).clip(0.03, 0.99)

    # Discounting penalty for longer exits
    discount_penalty = ((1 + disc_rate) ** (exit_time / 12.0) - 1).clip(0.0, 0.50)
    lgd = (lgd_base + 0.20 * discount_penalty).clip(0.03, 0.99)

    return lgd, exit_time

bridge['lgd_base'], bridge['exit_time_base'] = compute_bridging_lgd(bridge)

print(f"EAD-weighted base LGD: {exposure_weighted_average(bridge, 'lgd_base', 'ead'):.2%}")
display(bridge[['lgd_base', 'exit_time_base', 'exit_certainty', 'valuation_risk', 'failure_prob_base']].describe().round(4))



## 3. Delay Effect: Longer Exit -> Higher LGD


In [ ]:
bridge['exit_delay_bucket'] = pd.cut(
    bridge['exit_time_base'],
    bins=[0, 6, 9, 12, 18, 42],
    labels=['<=6m', '6-9m', '9-12m', '12-18m', '18m+'],
    right=True,
)

delay_rows = []
for b, grp in bridge.groupby('exit_delay_bucket', observed=True):
    delay_rows.append({
        'exit_delay_bucket': str(b),
        'loan_count': len(grp),
        'total_ead': grp['ead'].sum(),
        'ead_weighted_lgd_base': exposure_weighted_average(grp, 'lgd_base', 'ead'),
    })

delay_summary = pd.DataFrame(delay_rows).sort_values("ead_weighted_lgd_base").reset_index(drop=True)
display(delay_summary.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(delay_summary['exit_delay_bucket'], delay_summary['ead_weighted_lgd_base'], color='#dd8452')
ax.set_title('Delay Effect: Exit Time Bucket vs EAD-Weighted Base LGD')
ax.set_xlabel('Exit Time Bucket')
ax.set_ylabel('EAD-Weighted Base LGD')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'bridging_delay_vs_lgd.png'), dpi=140, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'bridging_delay_lgd_comparison.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
    print('Plot display skipped (set LGD_NOTEBOOK_SHOW_PLOTS=1 to show).')



## 4. Failed Exit Scenarios, Downturn LGD, and Final Overlay

Scenario design captures delay and failed-exit effects under stress.  
An explicit final overlay is then added to downturn LGD so cross-product comparison uses a consistent `lgd_final` output.


In [ ]:
scenario_specs = [
    {'scenario': 'base', 'delay_multiplier': 1.00, 'valuation_addon': 0.00, 'failure_addon': 0.00, 'rate_addon': 0.000},
    {'scenario': 'delay_stress', 'delay_multiplier': 1.30, 'valuation_addon': 0.02, 'failure_addon': 0.03, 'rate_addon': 0.005},
    {'scenario': 'failed_exit_stress', 'delay_multiplier': 1.45, 'valuation_addon': 0.04, 'failure_addon': 0.10, 'rate_addon': 0.010},
    {'scenario': 'combined_downturn', 'delay_multiplier': 1.60, 'valuation_addon': 0.07, 'failure_addon': 0.18, 'rate_addon': 0.020},
]

scenario_rows = []
for s in scenario_specs:
    lgd_s, exit_time_s = compute_bridging_lgd(
        bridge,
        delay_multiplier=s['delay_multiplier'],
        valuation_addon=s['valuation_addon'],
        failure_addon=s['failure_addon'],
        rate_addon=s['rate_addon'],
    )
    name = s['scenario']
    bridge[f'lgd_{name}'] = lgd_s
    bridge[f'exit_time_{name}'] = exit_time_s

    overlay_s = (
        0.010
        + 0.032 * bridge['failed_exit_flag']
        + 0.014 * bridge['valuation_risk']
        + 0.012 * ((exit_time_s - 9.0) / 18.0).clip(0.0, 1.0)
    ).clip(0.010, 0.090)
    bridge[f'final_overlay_addon_{name}'] = overlay_s
    bridge[f'lgd_final_{name}'] = (lgd_s + overlay_s).clip(0.0, 1.0)

    scenario_rows.append({
        'scenario': name,
        'ead_weighted_lgd_base': exposure_weighted_average(bridge, f'lgd_{name}', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(bridge, f'lgd_{name}', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(bridge, f'lgd_final_{name}', 'ead'),
        'mean_exit_time_months': float(exit_time_s.mean()),
    })

scenario_summary = pd.DataFrame(scenario_rows)
scenario_summary['mean_recovery_time_months'] = scenario_summary['mean_exit_time_months']
bridge['lgd_downturn'] = bridge['lgd_combined_downturn']
bridge['final_overlay_addon'] = bridge['final_overlay_addon_combined_downturn']
bridge['lgd_final'] = bridge['lgd_final_combined_downturn']

display(scenario_summary.round(4))

failed_vs_non = (
    bridge.groupby('failed_exit_flag', observed=True)
    .apply(
        lambda g: pd.Series({
            'loan_count': len(g),
            'total_ead': g['ead'].sum(),
            'ead_weighted_lgd_base': exposure_weighted_average(g, 'lgd_base', 'ead'),
            'ead_weighted_lgd_downturn': exposure_weighted_average(g, 'lgd_downturn', 'ead'),
            'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final', 'ead'),
        }),
        include_groups=False,
    )
    .reset_index()
)
display(failed_vs_non.round(4))



## 5. Weighted Outputs, Validation, and Exports


In [ ]:
segment_rows = []
for (purpose, final_exit), grp in bridge.groupby(['purpose', 'final_exit_type'], observed=True):
    segment_rows.append({
        'purpose': purpose,
        'final_exit_type': final_exit,
        'loan_count': len(grp),
        'total_ead': grp['ead'].sum(),
        'ead_weighted_lgd_base': exposure_weighted_average(grp, 'lgd_base', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(grp, 'lgd_downturn', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(grp, 'lgd_final', 'ead'),
        'mean_exit_certainty': grp['exit_certainty'].mean(),
        'mean_valuation_risk': grp['valuation_risk'].mean(),
        'mean_exit_time_base': grp['exit_time_base'].mean(),
    })

segment_summary = pd.DataFrame(segment_rows).sort_values(
    ['ead_weighted_lgd_final', 'total_ead'], ascending=[False, False]
).reset_index(drop=True)
display(segment_summary.round(4))

validation_checks = pd.DataFrame([
    {'check_name': 'lgd_base_between_0_and_1', 'passed': bool(bridge['lgd_base'].between(0, 1).all())},
    {'check_name': 'lgd_downturn_between_0_and_1', 'passed': bool(bridge['lgd_downturn'].between(0, 1).all())},
    {'check_name': 'lgd_final_between_0_and_1', 'passed': bool(bridge['lgd_final'].between(0, 1).all())},
    {'check_name': 'downturn_not_below_base', 'passed': bool((bridge['lgd_downturn'] >= bridge['lgd_base']).all())},
    {'check_name': 'final_not_below_downturn', 'passed': bool((bridge['lgd_final'] >= bridge['lgd_downturn']).all())},
    {'check_name': 'delay_stress_has_longer_exit_time', 'passed': bool((bridge['exit_time_combined_downturn'] >= bridge['exit_time_base']).all())},
])
display(validation_checks)

bridge.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'bridging_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'bridging_segment_summary.csv'), index=False)
delay_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'bridging_delay_summary.csv'), index=False)
scenario_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'bridging_scenario_summary.csv'), index=False)
validation_checks.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'bridging_validation_checks.csv'), index=False)

print('Saved bridging outputs:')
print('- bridging_loan_level_output.csv')
print('- bridging_segment_summary.csv')
print('- bridging_delay_summary.csv')
print('- bridging_scenario_summary.csv')
print('- bridging_validation_checks.csv')



## Assumptions and Limitations

- Bridging portfolio data is synthetic and for demonstration only.
- Exit certainty, valuation risk, and failure probability are proxy variables in absence of internal workout tapes.
- Downturn scenarios are designed to transparently show how delayed/failed exits increase LGD.


---

## APS 113 Calibration Layer — Bridging Loans

> **SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY**
>
> This section adds a full APS 113-aligned calibration loop on top of the
> proxy baseline above. Workout data is synthetically generated (2014-2024,
> 10-year window) to demonstrate methodology; real production calibration
> requires an internal workout tape.

### Calibration Pipeline (APS 113 s.32-68)

| Step | Function | APS 113 |
|------|----------|---------|
| 1 | Load/generate historical workout data | s.44, Att A |
| 2 | `compute_realised_lgd()` — LIP costs, cure leg | s.32-34 |
| 3 | `classify_economic_regime()` + `assign_regime_to_workouts()` | s.43, s.46 |
| 4 | `segment_lgd()` — product-specific segment keys | s.52 |
| 5 | `compute_long_run_lgd()` — vintage-EWA method | s.43-44 |
| 6 | `compare_model_vs_actual()` — proxy vs realised | s.60-62 |
| 7 | `apply_downturn_overlay()` + `apply_correlation_adjustment()` | s.46-50, s.55-57 |
| 8 | `MoCRegister` + `apply_moc()` — 5 APS 113 s.65 sources | s.63-65 |
| 9 | `apply_regulatory_floor()` — 25% per APS 113 s.58 | s.58 |
| 10 | Export 9 CSV outputs | — |
| 11 | `run_full_validation_suite()` — Gini, HL, PSI, OOT | s.66-68 |

**Correct APS 113 order:** LR-LGD → downturn overlay → correlation adj →
MoC → floor (MoC applied to downturn LGD, not long-run LGD, per s.63).


In [ ]:
# APS 113 Calibration Layer — imports and configuration
import os, sys
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))

from src.calibration_utils import (
    compute_realised_lgd,
    segment_lgd,
    compute_long_run_lgd,
    compare_model_vs_actual,
    classify_economic_regime,
    assign_regime_to_workouts,
    apply_downturn_overlay,
    apply_correlation_adjustment,
    build_lgd_pd_annual_series,
    estimate_lgd_pd_correlation,
    MoCRegister,
    apply_moc,
    apply_regulatory_floor,
    run_full_validation_suite,
    generate_compliance_map,
    export_compliance_map,
    CALIBRATION_STEP_ORDER,
)
from src.generators import GENERATOR_MAP, generate_all_historical_workouts

PRODUCT = "bridging"
SEGMENT_KEYS = ['exit_certainty_band', 'exit_type']
MODEL_LGD_COL = "lgd_final"

HISTORY_DIR = Path('..') / 'data' / 'generated' / 'historical'
OUTPUTS_DIR = Path('..') / 'outputs' / 'tables'
UPSTREAM_PARQUET = Path('..') / 'data' / 'exports' / 'macro_regime_flags.parquet'

HISTORY_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Product: {PRODUCT}")
print(f"Segment keys: {SEGMENT_KEYS}")
print(f"APS 113 calibration pipeline — step order:")
for step, fn, ref in CALIBRATION_STEP_ORDER:
    print(f"  Step {step:>2}: {fn:<35} {ref}")


In [ ]:
# ── STEP 1: Load or generate historical workout data (APS 113 s.44, Att A) ─
parquet_path = HISTORY_DIR / f"{PRODUCT}_workouts.parquet"

if parquet_path.exists():
    cal_loans = pd.read_parquet(parquet_path)
    # cashflows stored alongside or re-generated
    try:
        cal_cashflows = pd.read_parquet(
            HISTORY_DIR / f"{PRODUCT}_cashflows.parquet"
        )
    except FileNotFoundError:
        cal_cashflows = None
    print(f"Loaded {len(cal_loans):,} historical workout loans from Parquet")
else:
    print(f"Parquet not found — generating synthetic workout data for {PRODUCT}")
    results = generate_all_historical_workouts(
        seed=42, output_dir=HISTORY_DIR, write_parquet=True, products=[PRODUCT]
    )
    cal_loans = results[PRODUCT]["loans"]
    cal_cashflows = results[PRODUCT].get("cashflows")
    print(f"Generated {len(cal_loans):,} synthetic workout loans (2014-2024)")

print(f"Date range: {cal_loans['default_date'].min()} to {cal_loans['default_date'].max()}")
print(f"Columns: {list(cal_loans.columns)}")

# ── STEP 2: Compute realised LGD (APS 113 s.32-34) ─────────────────────────
# LIP costs (Loss Identification Period, ~90 days) auto-detected if cashflows available
lgd_df = compute_realised_lgd(
    loans=cal_loans,
    cashflows=cal_cashflows,
    ead_col="ead_at_default",
    recovery_col="recovery_amount",
    cost_col="direct_costs",
    lip_window_days=90,
)
print(f"\nStep 2: Realised LGD computed")
print(f"  EAD-weighted realised LGD: {(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum():.2%}")
display(lgd_df[['realised_lgd', 'ead_at_default']].describe().round(4))

# ── STEP 3: Economic regime classification (APS 113 s.43, s.46-50) ─────────
upstream_path = str(UPSTREAM_PARQUET) if UPSTREAM_PARQUET.exists() else None
regime_df = classify_economic_regime(
    upstream_parquet_path=upstream_path,
    method="upstream_first",
)
print(f"\nStep 3: Economic regimes classified")
print(f"  Data source: {regime_df['data_source'].iloc[0]}")
display(regime_df[['year', 'regime', 'is_downturn_period', 'data_source']].head(12))

lgd_df = assign_regime_to_workouts(lgd_df, regime_df)
downturn_pct = lgd_df.get('is_downturn_period', pd.Series([False])).mean()
print(f"  Downturn observations: {downturn_pct:.1%} of portfolio")

# ── STEP 4: Segment by product-specific keys (APS 113 s.52) ────────────────
segmented_df = segment_lgd(lgd_df, SEGMENT_KEYS)
low_count = segmented_df[segmented_df.get('segment_flag', '') == 'low_count'] if 'segment_flag' in segmented_df.columns else pd.DataFrame()
print(f"\nStep 4: Segmentation complete")
print(f"  Segments: {segmented_df.groupby(SEGMENT_KEYS, observed=True).ngroups}")
if not low_count.empty:
    print(f"  Low-count segments flagged (<20 obs): {len(low_count)}")

# ── STEP 5: Long-run LGD — vintage-EWA (APS 113 s.43-44) ─────────────────
lr_lgd_df = compute_long_run_lgd(
    segmented_df,
    segment_keys=SEGMENT_KEYS,
    method="vintage_ewa",
)
print(f"\nStep 5: Long-run LGD by segment (vintage-EWA)")
display(lr_lgd_df.round(4))
lr_lgd_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_long_run_lgd_by_segment.csv", index=False)

# ── STEP 6: Compare model vs actual (APS 113 s.60-62) ──────────────────────
# 'model_lgd' here = proxy model LGD from the section above (lgd_final if present)
if MODEL_LGD_COL in cal_loans.columns:
    compare_input = lgd_df.merge(
        cal_loans[['loan_id', MODEL_LGD_COL] if 'loan_id' in cal_loans.columns else [MODEL_LGD_COL]],
        left_index=True, right_index=True, how='left',
    ) if MODEL_LGD_COL not in lgd_df.columns else lgd_df
else:
    compare_input = lgd_df.copy()
    compare_input['model_lgd'] = lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else 0.25

model_col_to_use = MODEL_LGD_COL if MODEL_LGD_COL in compare_input.columns else 'model_lgd'
mva_df = compare_model_vs_actual(
    compare_input,
    model_lgd_col=model_col_to_use,
    segment_keys=SEGMENT_KEYS,
)
print(f"\nStep 6: Model vs actual comparison")
display(mva_df.round(4))
mva_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_model_vs_actual_comparison.csv", index=False)

# ── STEP 7: Downturn overlay + Frye-Jacobs correlation adj (s.46-50, s.55-57)
# Reuses apply_downturn_overlay from existing lgd_calculation.py
dt_lgd = apply_downturn_overlay(segmented_df, product=PRODUCT)
print(f"\nStep 7: Downturn overlay applied")
if 'lgd_downturn' in dt_lgd.columns:
    ewa_dt = (dt_lgd['lgd_downturn'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    ewa_lr = (dt_lgd['realised_lgd'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    print(f"  EWA Long-run LGD:  {ewa_lr:.2%}")
    print(f"  EWA Downturn LGD:  {ewa_dt:.2%}")
    downturn_col = 'lgd_downturn'
else:
    dt_lgd['lgd_downturn'] = dt_lgd['realised_lgd'] * 1.15  # fallback scalar
    downturn_col = 'lgd_downturn'

# Frye-Jacobs correlation adjustment (APS 113 s.55-57)
try:
    lgd_ts, pd_ts = build_lgd_pd_annual_series(dt_lgd)
    macro_for_corr = regime_df.rename(columns={'gdp_growth': 'gdp_growth_yoy'})
    corr_result = estimate_lgd_pd_correlation(lgd_ts, pd_ts, macro_for_corr)
    dt_lgd['lgd_downturn_corr_adj'] = apply_correlation_adjustment(
        dt_lgd[downturn_col], corr_result['rho'], corr_result['macro_shock_std']
    )
    downturn_col = 'lgd_downturn_corr_adj'
    print(f"  Frye-Jacobs rho={corr_result['rho']:.3f}, adj factor={corr_result['lgd_dt_adjustment_factor']:.3f}")
except Exception as exc:
    print(f"  Frye-Jacobs skipped: {exc}")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_downturn_lgd_by_segment.csv", index=False)

# ── STEP 8: MoC register + apply MoC (AFTER downturn — APS 113 s.63-65) ───
# Determine regime data source for MoC model_approximation component
data_src = regime_df['data_source'].iloc[0] if 'data_source' in regime_df.columns else 'synthetic'
n_downturn_yrs = int(regime_df['is_downturn_period'].sum()) if 'is_downturn_period' in regime_df.columns else 0

psi_approx = 0.05  # placeholder — full PSI computed in Step 11
bias_approx = float(mva_df['bias'].abs().mean()) if 'bias' in mva_df.columns else 0.02

moc_register = MoCRegister(product=PRODUCT, regime_data_source=data_src)
moc_df = moc_register.build_moc_register(
    segment_df=segmented_df,
    segment_keys=SEGMENT_KEYS,
    n_downturn_vintages=n_downturn_yrs,
    psi_value=psi_approx,
    backtesting_bias=bias_approx,
)
print(f"\nStep 8: MoC register built")
display(moc_df.round(4))
moc_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_moc_register.csv", index=False)

lgd_with_moc = apply_moc(dt_lgd[downturn_col], moc_df, segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None)
dt_lgd['lgd_with_moc'] = lgd_with_moc
moc_ewa = (lgd_with_moc * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
print(f"  EWA LGD after MoC: {moc_ewa:.2%}")

# ── STEP 9: Regulatory floors (APS 113 s.58) ──────────────────────────────
dt_lgd['lgd_final_calibrated'] = apply_regulatory_floor(dt_lgd['lgd_with_moc'], product=PRODUCT)
final_ewa = (dt_lgd['lgd_final_calibrated'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
floor_binding_pct = (dt_lgd['lgd_final_calibrated'] > dt_lgd['lgd_with_moc']).mean()
print(f"\nStep 9: Regulatory floor applied")
print(f"  EWA Final Calibrated LGD: {final_ewa:.2%}")
print(f"  Floor binding for: {floor_binding_pct:.1%} of loans")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_final_calibrated_lgd.csv", index=False)

# ── STEP 10: Export all outputs ────────────────────────────────────────────
# Already exported: long_run_lgd_by_segment, model_vs_actual, downturn_lgd, moc_register, final_calibrated_lgd
# Export remaining:
lgd_df[['realised_lgd', 'ead_at_default']].assign(product=PRODUCT).to_csv(
    OUTPUTS_DIR / f"{PRODUCT}_historical_workouts.csv", index=False
)
regime_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_regime_classification.csv", index=False)

# Calibration adjustments summary
cal_adj_summary = pd.DataFrame({
    'product': [PRODUCT],
    'ewa_realised_lgd': [(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum()],
    'ewa_long_run_lgd': [lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None],
    'ewa_downturn_lgd': [(dt_lgd.get('lgd_downturn', dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_with_moc': [(dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_final': [final_ewa],
    'floor_binding_pct': [floor_binding_pct],
    'regime_data_source': [data_src],
    'n_loans': [len(lgd_df)],
    'calibration_date': [pd.Timestamp.today().date()],
})
cal_adj_summary.to_csv(OUTPUTS_DIR / f"{PRODUCT}_calibration_adjustments.csv", index=False)
print(f"\nStep 10: All outputs exported to {OUTPUTS_DIR}")

# ── STEP 11: Full validation suite (APS 113 s.66-68) ──────────────────────
print(f"\nStep 11: Running full validation suite...")
try:
    val_results = run_full_validation_suite(
        loans=dt_lgd,
        predicted_col='lgd_final_calibrated',
        actual_col='realised_lgd',
        segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None,
        date_col='default_date' if 'default_date' in dt_lgd.columns else None,
        product=PRODUCT,
    )
    print(f"  Gini: {val_results.get('gini', 'n/a')}")
    print(f"  Calibration ratio: {val_results.get('calibration_ratio', 'n/a')}")
    print(f"  PSI: {val_results.get('psi', 'n/a')}")
    if 'summary_table' in val_results:
        val_results['summary_table'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_validation_report.csv", index=False
        )
    if 'backtest_results' in val_results:
        val_results['backtest_results'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_backtest_results.csv", index=False
        )
except Exception as exc:
    print(f"  Validation suite error (non-fatal): {exc}")


In [ ]:
# ── Calibration summary waterfall ──────────────────────────────────────────
import matplotlib.pyplot as plt

try:
    stages = {
        'Realised LGD\n(2014-2024)': (lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum(),
        'Long-run LGD\n(vintage-EWA)': lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None,
        'Downturn LGD': (dt_lgd.get(downturn_col, dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        '+ MoC': (dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        'Final\n(Floor Applied)': final_ewa,
    }
    labels = [k for k, v in stages.items() if v is not None]
    values = [v for v in stages.values() if v is not None]
    colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#8e44ad'][:len(labels)]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_ylabel('EAD-Weighted LGD')
    ax.set_title(f'APS 113 Calibration Waterfall — Bridging Loans')
    ax.set_ylim(0, max(values) * 1.35)
    ax.axhline(values[-1], color='black', ls=':', lw=1, label=f'Final: {values[-1]:.1%}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        Path('..') / 'outputs' / 'figures' / f'bridging_calibration_waterfall.png',
        dpi=150, bbox_inches='tight',
    )
    plt.show()
except Exception as exc:
    print(f"Waterfall chart error (non-fatal): {exc}")

# ── APS 113 compliance snapshot ─────────────────────────────────────────────
compliance_df = generate_compliance_map(
    calibration_results={PRODUCT: {'long_run_lgd_by_segment': True, 'calibration_steps': True}},
    moc_registers={PRODUCT: moc_df},
    regime_data_source=data_src,
    products=[PRODUCT],
)
print("\n=== APS 113 Compliance Snapshot ===")
display(compliance_df[['section_ref', 'requirement', 'status', 'reviewer_note']].set_index('section_ref'))
export_compliance_map(compliance_df, OUTPUTS_DIR / f"bridging_aps113_compliance.csv")

# Final summary
print("\n=== Calibration Summary ===")
display(cal_adj_summary.round(4))
print(f"\nAll calibration outputs in: {OUTPUTS_DIR}")
print(f"SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY")
